# 🏥 Fine-tune BamiBERT Medical NER — Approach 3: Two-Stage
### Stage 1: Continued MLM Pretraining → Stage 2: BIO NER Fine-tuning
> **Chạy tuần tự từ Cell 1 → Cell 7.** Kernel cần GPU A10G hoặc T4, RAM ≥ 16 GiB.
>
> **Kết quả:** Model được upload lên HuggingFace Hub repo mới, tích hợp trực tiếp vào `MedicalNEREncoder`.

In [ ]:
# ============================================================
# CELL 1 - Cài đặt thư viện
# Modal.ai dùng uv package manager → dùng magic %uv pip
# ============================================================
%uv pip install --upgrade transformers==5.5.0 datasets==4.3.0 evaluate accelerate huggingface_hub seqeval

print("✅ Cài đặt hoàn tất!")

In [ ]:
# ============================================================
# CELL 2 - Cấu hình: HF Token, Repo, Đường dẫn dữ liệu
# ============================================================
import os

# ⚠️ ĐIỀN TOKEN CỦA BẠN VÀO ĐÂY
# Lấy tại: https://huggingface.co/settings/tokens (cần quyền Write)
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"   # <-- ĐỔI TOKEN

# Tên repo mới sẽ được tạo trên HuggingFace
HF_REPO_NAME = "BamiBERT-ViMedNER-Finetuned"          # <-- ĐỔI TÊN NẾU MUỐN

# Model gốc (pretrained, chưa finetune)
BASE_MODEL_NAME = "cbc-528a/BamiBERT-ViMedNER"

# Đường dẫn dữ liệu trên Modal
# Upload toàn bộ folder data/ vào /root/data trước khi chạy
DATA_BASE         = "/root/data"
TRAIN_TEXT_DIR    = os.path.join(DATA_BASE, "train")              # 100 files txt không nhãn
FINETUNE_TEXT_DIR = os.path.join(DATA_BASE, "finetune", "text")   # 26 files txt có nhãn
FINETUNE_GT_DIR   = os.path.join(DATA_BASE, "finetune", "gt")     # 26 files json GT

# Thư mục lưu model
STAGE1_OUTPUT = "/root/bami_adapted_mlm"
STAGE2_OUTPUT = "/root/bami_ner_finetuned"

# Hyperparameters
STAGE1_EPOCHS = 5
STAGE2_EPOCHS = 20
BATCH_SIZE    = 8
MAX_SEQ_LEN   = 512

print("✅ Cấu hình:")
print(f"   Base model  : {BASE_MODEL_NAME}")
print(f"   HF Repo     : {HF_REPO_NAME}")
print(f"   Train dir   : {TRAIN_TEXT_DIR}")
print(f"   Finetune dir: {FINETUNE_TEXT_DIR}")
print(f"   Stage1 out  : {STAGE1_OUTPUT}")
print(f"   Stage2 out  : {STAGE2_OUTPUT}")

In [ ]:
# ============================================================
# CELL 3 - Stage 1: Continued MLM Pretraining
# Học thêm trên 100 docs data/train/ (không cần nhãn)
# Mục tiêu: BamiBERT hiểu sâu hơn văn phong hồ sơ bệnh viện
# ============================================================
import glob
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── 3.1 Load tokenizer ─────────────────────────────────────
print(f"\n[Stage 1] Đang tải tokenizer từ {BASE_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

# ── 3.2 Load 100 unlabeled docs ────────────────────────────
train_txt_files = sorted(glob.glob(os.path.join(TRAIN_TEXT_DIR, "*.txt")))
print(f"[Stage 1] Tìm thấy {len(train_txt_files)} files trong data/train/")

raw_texts = []
for fpath in train_txt_files:
    with open(fpath, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if content:
            raw_texts.append(content)

total_chars = sum(len(t) for t in raw_texts)
print(f"[Stage 1] Đọc xong {len(raw_texts)} docs (tổng ~{total_chars:,} ký tự)")

# ── 3.3 Tokenize ───────────────────────────────────────────
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,  # DataCollatorForLanguageModeling sẽ pad
    )

raw_dataset = Dataset.from_dict({"text": raw_texts})
tokenized_dataset = raw_dataset.map(
    tokenize_function, batched=True,
    remove_columns=["text"], desc="Tokenizing Stage 1",
)

# DataCollator tự động che ngẫu nhiên 15% tokens (MLM)
data_collator_mlm = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15,
)

# ── 3.4 Load model ở chế độ MLM ───────────────────────────
print(f"\n[Stage 1] Đang tải {BASE_MODEL_NAME} ở chế độ AutoModelForMaskedLM...")
model_mlm = AutoModelForMaskedLM.from_pretrained(BASE_MODEL_NAME)
num_params = model_mlm.num_parameters()
print(f"[Stage 1] Số tham số: {num_params:,}")

# ── 3.5 TrainingArguments ──────────────────────────────────
stage1_args = TrainingArguments(
    output_dir=STAGE1_OUTPUT,
    num_train_epochs=STAGE1_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    save_strategy="epoch",
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to="none",
    push_to_hub=False,
)

# ── 3.6 Trainer & Train ────────────────────────────────────
trainer_mlm = Trainer(
    model=model_mlm,
    args=stage1_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator_mlm,
    processing_class=tokenizer,
)

print(f"\n🚀 [Stage 1] Bắt đầu Continued MLM Pretraining...")
trainer_mlm.train()

# ── 3.7 Lưu model Stage 1 ──────────────────────────────────
trainer_mlm.save_model(STAGE1_OUTPUT)
tokenizer.save_pretrained(STAGE1_OUTPUT)
print(f"\n✅ [Stage 1] Hoàn tất! Model đã lưu tại: {STAGE1_OUTPUT}")

In [ ]:
# ============================================================
# CELL 4 - Chuẩn bị BIO Dataset từ 26 GT files
# Chuyển đổi GT char-offset → BIO token labels dùng offset_mapping
# ============================================================
import json
import random
import numpy as np
from collections import Counter
from datasets import DatasetDict

# Reload tokenizer từ Stage 1
tokenizer = AutoTokenizer.from_pretrained(STAGE1_OUTPUT)

# ── Định nghĩa label set ───────────────────────────────────
ENTITY_TYPES = [
    "CHẨN_ĐOÁN",
    "THUỐC",
    "TRIỆU_CHỨNG",
    "TÊN_XÉT_NGHIỆM",
    "KẾT_QUẢ_XÉT_NGHIỆM",
]
LABELS = ["O"]
for etype in ENTITY_TYPES:
    LABELS.append(f"B-{etype}")
    LABELS.append(f"I-{etype}")

label2id = {lbl: i for i, lbl in enumerate(LABELS)}
id2label = {i: lbl for i, lbl in enumerate(LABELS)}

print(f"Label set ({len(LABELS)} labels):")
for i, lbl in enumerate(LABELS):
    print(f"  {i:2d}: {lbl}")

# ── Hàm convert GT → BIO token labels ────────────────────
def convert_gt_to_bio_tokens(text, entities, tokenizer, max_length=512):
    """
    Chuyển đổi char-offset GT sang BIO token labels.
    Dùng offset_mapping để alignment chính xác với subword tokenizer.
    """
    # Bước 1: Char-level label array
    char_labels = ["O"] * len(text)
    for ent in entities:
        pos   = ent.get("position")
        etype = ent.get("type", "")
        if not pos or len(pos) < 2 or etype not in ENTITY_TYPES:
            continue
        start, end = pos[0], pos[1]
        if start >= len(text) or end > len(text) or start >= end:
            continue
        char_labels[start] = f"B-{etype}"
        for ci in range(start + 1, end):
            char_labels[ci] = f"I-{etype}"

    # Bước 2: Tokenize với offset_mapping
    encoding = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # Bước 3: Map từng token → label
    # offset_mapping: [(tok_start, tok_end), ...]
    # Special tokens [CLS]/[SEP]/[PAD] → (0,0) → label = -100 (ignored in loss)
    token_labels = []
    for tok_start, tok_end in encoding["offset_mapping"]:
        if tok_start == tok_end:
            token_labels.append(-100)
        else:
            token_labels.append(label2id[char_labels[tok_start]])

    return {
        "input_ids":      encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels":         token_labels,
    }

# ── Load 26 GT files ──────────────────────────────────────
gt_files = sorted(glob.glob(os.path.join(FINETUNE_GT_DIR, "*.json")))
print(f"\nTìm thấy {len(gt_files)} file GT trong data/finetune/gt/")

all_samples, skipped = [], 0
for gt_path in gt_files:
    basename = os.path.basename(gt_path)
    # Xử lý cả "1.json" và "7_labels.json"
    stem     = basename.replace("_labels", "").replace(".json", "")
    txt_path = os.path.join(FINETUNE_TEXT_DIR, f"{stem}.txt")

    if not os.path.exists(txt_path):
        print(f"  ⚠️  Không tìm thấy: {txt_path}")
        skipped += 1
        continue

    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read().strip()
    with open(gt_path, "r", encoding="utf-8") as f:
        entities = json.load(f)

    if not text or not entities:
        skipped += 1
        continue

    sample = convert_gt_to_bio_tokens(text, entities, tokenizer, MAX_SEQ_LEN)
    all_samples.append(sample)
    print(f"  ✅ {basename} → {len(entities)} entities")

print(f"\nĐã xử lý {len(all_samples)} samples, bỏ qua {skipped}")

# ── Thống kê label distribution ──────────────────────────
label_counts = Counter()
for s in all_samples:
    for lid in s["labels"]:
        if lid != -100:
            label_counts[id2label[lid]] += 1
print("\n📊 Phân bố labels:")
for lbl, cnt in sorted(label_counts.items()):
    print(f"  {lbl:<40}: {cnt:>5}")

# ── Train/Val split 80/20 ────────────────────────────────
random.seed(42)
random.shuffle(all_samples)
split_idx     = max(1, int(len(all_samples) * 0.8))
train_samples = all_samples[:split_idx]
val_samples   = all_samples[split_idx:]
print(f"\nTrain: {len(train_samples)} | Val: {len(val_samples)}")

def to_hf_dataset(samples):
    return Dataset.from_dict({
        "input_ids":      [s["input_ids"]      for s in samples],
        "attention_mask": [s["attention_mask"] for s in samples],
        "labels":         [s["labels"]         for s in samples],
    })

bio_dataset = DatasetDict({
    "train":      to_hf_dataset(train_samples),
    "validation": to_hf_dataset(val_samples),
})
print(f"\n✅ BIO Dataset sẵn sàng: {bio_dataset}")

In [ ]:
# ============================================================
# CELL 5 - Stage 2: BIO NER Fine-tuning từ model Stage 1
# ============================================================
import evaluate
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
)

# ── 5.1 Load model từ Stage 1 (backbone đã adapted) ──────
print(f"[Stage 2] Đang tải model từ Stage 1 ({STAGE1_OUTPUT})...")
model_ner = AutoModelForTokenClassification.from_pretrained(
    STAGE1_OUTPUT,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,  # Reset MLM head → NER Linear head mới
)
num_params = model_ner.num_parameters()
print(f"[Stage 2] Số tham số: {num_params:,} | Labels: {len(LABELS)}")

# ── 5.2 DataCollator cho Token Classification ────────────
data_collator_ner = DataCollatorForTokenClassification(
    tokenizer=tokenizer, label_pad_token_id=-100,
)

# ── 5.3 Metrics: seqeval (entity-level F1, chuẩn CoNLL) ──
seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)

    true_labels_batch, pred_labels_batch = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        true_seq, pred_seq_f = [], []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:  # Bỏ qua special/pad tokens
                true_seq.append(id2label[l])
                pred_seq_f.append(id2label[p])
        true_labels_batch.append(true_seq)
        pred_labels_batch.append(pred_seq_f)

    results = seqeval_metric.compute(
        predictions=pred_labels_batch,
        references=true_labels_batch,
    )
    output = {
        "precision": results["overall_precision"],
        "recall"   : results["overall_recall"],
        "f1"       : results["overall_f1"],
        "accuracy" : results["overall_accuracy"],
    }
    # F1 per entity type
    for etype in ENTITY_TYPES:
        if etype in results:
            output[f"f1_{etype}"] = results[etype]["f1"]
    return output

# ── 5.4 TrainingArguments ─────────────────────────────────
stage2_args = TrainingArguments(
    output_dir=STAGE2_OUTPUT,
    num_train_epochs=STAGE2_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=1e-5,           # Thấp hơn Stage 1 để tránh overfit (26 samples)
    weight_decay=0.01,
    warmup_ratio=0.15,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=5,
    report_to="none",
    push_to_hub=False,
    max_grad_norm=1.0,           # Gradient clipping để ổn định training
)

# ── 5.5 Trainer ───────────────────────────────────────────
trainer_ner = Trainer(
    model=model_ner,
    args=stage2_args,
    train_dataset=bio_dataset["train"],
    eval_dataset=bio_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator_ner,
    compute_metrics=compute_metrics,
)

n_train = len(bio_dataset["train"])
n_val   = len(bio_dataset["validation"])
print(f"\n🚀 [Stage 2] Bắt đầu NER Fine-tuning...")
print(f"   Train samples: {n_train} | Val samples: {n_val}")
trainer_ner.train()

# ── 5.6 Lưu best model (load_best_model_at_end=True) ─────
trainer_ner.save_model(STAGE2_OUTPUT)
tokenizer.save_pretrained(STAGE2_OUTPUT)
print(f"\n✅ [Stage 2] Hoàn tất! Model đã lưu tại: {STAGE2_OUTPUT}")

In [ ]:
# ============================================================
# CELL 6 - Đánh giá chi tiết & Demo Inference
# ============================================================
from transformers import AutoModelForTokenClassification, pipeline as hf_pipeline

# Đánh giá trên tập Val
print("📊 Kết quả đánh giá trên tập Validation:")
eval_results = trainer_ner.evaluate()
print()
print(f"  Overall Precision : {eval_results.get('eval_precision', 0):.4f}")
print(f"  Overall Recall    : {eval_results.get('eval_recall', 0):.4f}")
print(f"  Overall F1        : {eval_results.get('eval_f1', 0):.4f}")
print()
print("  F1 per entity type:")
for etype in ENTITY_TYPES:
    key = f"eval_f1_{etype}"
    if key in eval_results:
        print(f"    {etype:<35}: {eval_results[key]:.4f}")

# Demo inference
print("\n🧪 Demo inference:")
model_eval = AutoModelForTokenClassification.from_pretrained(STAGE2_OUTPUT)
eval_tok   = AutoTokenizer.from_pretrained(STAGE2_OUTPUT)
ner_pipe   = hf_pipeline(
    "ner", model=model_eval, tokenizer=eval_tok, aggregation_strategy="simple",
)

test_sentences = [
    "Điều trị tăng huyết áp và đái tháo đường típ 2 bằng gleevec.",
    "Kết quả CEA tăng nhẹ 4.9, siêu âm bụng âm tính.",
    "Bệnh nhân không có đau ngực, không ho, không sốt.",
]

for sent in test_sentences:
    print(f"\n  Input: {sent}")
    for r in ner_pipe(sent):
        eg  = r["entity_group"]
        wd  = r["word"]
        sc  = r["score"]
        print(f"    → [{eg}] '{wd}' (score={sc:.3f})")

In [ ]:
# ============================================================
# CELL 7 - Upload lên HuggingFace Hub (repo mới)
# ============================================================
from huggingface_hub import HfApi, login
from transformers import AutoModelForTokenClassification, AutoTokenizer

# Đăng nhập
print("🔐 Đăng nhập HuggingFace...")
login(token=HF_TOKEN)

api         = HfApi()
user_info   = api.whoami(token=HF_TOKEN)
HF_USERNAME = user_info["name"]
FULL_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"
print(f"✅ Đăng nhập thành công. Username: {HF_USERNAME}")
print(f"📦 Repo: https://huggingface.co/{FULL_REPO_ID}")

# Tạo repo mới (exist_ok=True → không lỗi nếu đã tồn tại)
api.create_repo(
    repo_id=FULL_REPO_ID,
    token=HF_TOKEN,
    repo_type="model",
    exist_ok=True,
    private=False,
)
print(f"✅ Repo '{FULL_REPO_ID}' đã sẵn sàng.")

# Load model tốt nhất và push
print(f"\n📤 Đang upload model từ {STAGE2_OUTPUT}...")
final_model     = AutoModelForTokenClassification.from_pretrained(STAGE2_OUTPUT)
final_tokenizer = AutoTokenizer.from_pretrained(STAGE2_OUTPUT)

final_model.push_to_hub(
    FULL_REPO_ID,
    token=HF_TOKEN,
    commit_message="Upload BamiBERT NER finetuned — Approach 3: Two-Stage MLM + BIO",
)
final_tokenizer.push_to_hub(FULL_REPO_ID, token=HF_TOKEN)

print(f"\n🎉 Upload thành công!")
print(f"   Model URL: https://huggingface.co/{FULL_REPO_ID}")
print()
print("📋 Để dùng model mới trong pipeline.py, chỉ cần đổi 1 dòng:")
print(f'   MedicalNEREncoder(model_name="{FULL_REPO_ID}")')
print()
print("⚠️  Lưu ý thêm: Trong MedicalNEREncoder.extract(), xóa bỏ bước")
print("   label_mapping vì model mới output thẳng labels tiếng Việt:")
print("   CHẨN_ĐOÁN, THUỐC, TRIỆU_CHỨNG, TÊN_XÉT_NGHIỆM, KẾT_QUẢ_XÉT_NGHIỆM")